In [ ]:
%pip install matplotlib

In [ ]:
%pip install pyarrow


In [ ]:
# ============================================================
# 01 — IMPORTS ET CONFIGURATION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# Affichage pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

# Reproductibilité / affichage
RANDOM_STATE = 42

# Répertoire du dataset
DATA_DIR = Path("../data/financial-fraud")

# Fichiers
TRANSACTIONS_FILE = DATA_DIR / "transactions_data.csv"
CARDS_FILE = DATA_DIR / "cards_data.csv"
USERS_FILE = DATA_DIR / "users_data.csv"
MCC_FILE = DATA_DIR / "mcc_codes.json"
LABELS_FILE = DATA_DIR / "train_fraud_labels.json"

print("Configuration chargée.")
print(f"Dataset : {DATA_DIR.resolve()}")

In [ ]:
# ============================================================
# 02 — CHARGEMENT DES DONNÉES
# ============================================================

transactions = pd.read_csv(TRANSACTIONS_FILE)
cards = pd.read_csv(CARDS_FILE)
users = pd.read_csv(USERS_FILE)
labels = pd.read_json(LABELS_FILE)

print("Données chargées avec succès.\n")

print(f"Transactions : {transactions.shape}")
print(f"Cards        : {cards.shape}")
print(f"Users        : {users.shape}")
print(f"Labels       : {labels.shape}")

In [ ]:
# Préparation des labels
labels["is_fraud"] = (
    labels["target"]
    .map({"Yes": 1, "No": 0})
    .astype("int8")
)

# L'index du fichier labels correspond à l'identifiant de transaction
labels.index.name = "id"

print("\nRépartition des labels :")
print(labels["is_fraud"].value_counts())

print("\nPourcentage :")
print(
    labels["is_fraud"]
    .value_counts(normalize=True)
    .mul(100)
    .round(4)
)

In [ ]:
# ============================================================
# 03 — CONTRÔLE INITIAL DES DONNÉES
# ============================================================

print("=== COLONNES ===\n")

print("Transactions :")
print(transactions.columns.tolist())

print("\nCards :")
print(cards.columns.tolist())

print("\nUsers :")
print(users.columns.tolist())

print("\nLabels :")
print(labels.columns.tolist())


print("\n=== DIMENSIONS ===\n")

print(f"Transactions : {transactions.shape}")
print(f"Cards        : {cards.shape}")
print(f"Users        : {users.shape}")
print(f"Labels       : {labels.shape}")


print("\n=== IDENTIFIANTS ===\n")

print(
    "IDs transactions uniques :",
    transactions["id"].nunique()
)

print(
    "IDs labels uniques       :",
    labels.index.nunique()
)

print(
    "IDs cartes uniques       :",
    transactions["card_id"].nunique()
)

print(
    "IDs clients uniques      :",
    transactions["client_id"].nunique()
)

print(
    "IDs marchands uniques    :",
    transactions["merchant_id"].nunique()
)

In [ ]:
# Vérification de l'intégrité des clés de référence

print("\n=== INTÉGRITÉ DES CLÉS ===\n")

missing_card_ids = (
    ~transactions["card_id"].isin(cards["id"])
).sum()

missing_client_ids = (
    ~transactions["client_id"].isin(users["id"])
).sum()

print(
    "Transactions avec card_id absent de cards :",
    missing_card_ids
)

print(
    "Transactions avec client_id absent de users :",
    missing_client_ids
)

In [ ]:
# ============================================================
# 04 — PRÉPARATION DE LA TABLE TRANSACTIONNELLE
# ============================================================

transaction_columns = [
    "id",
    "date",
    "client_id",
    "card_id",
    "amount",
    "use_chip",
    "merchant_id",
    "merchant_city",
    "merchant_state",
    "zip",
    "mcc",
    "errors"
]

transactions_eda = transactions[transaction_columns].copy()

# ------------------------------------------------------------
# Conversion de la date
# ------------------------------------------------------------

transactions_eda["date"] = pd.to_datetime(
    transactions_eda["date"],
    errors="coerce"
)

# ------------------------------------------------------------
# Conversion du montant
# ------------------------------------------------------------

transactions_eda["amount_numeric"] = (
    transactions_eda["amount"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# ------------------------------------------------------------
# Indicateur de présence d'une erreur
# ------------------------------------------------------------

transactions_eda["has_error"] = (
    transactions_eda["errors"]
    .notna()
    .astype("int8")
)

print("Table transactionnelle préparée.")
print("Dimensions :", transactions_eda.shape)
print("\nColonnes :")
print(transactions_eda.columns.tolist())

In [ ]:
# ============================================================
# 05 — TABLE TRANSACTIONNELLE LABELLISÉE
# ============================================================

# Préparation d'une table de labels indexée par l'identifiant
# de transaction
labels_eda = labels[["is_fraud"]].copy()
labels_eda.index.name = "id"

# ------------------------------------------------------------
# Jointure transactions ↔ labels
# ------------------------------------------------------------

labeled_transactions = transactions_eda.merge(
    labels_eda,
    left_on="id",
    right_index=True,
    how="inner",
    validate="one_to_one"
)

print("Table labellisée créée.")
print("Dimensions :", labeled_transactions.shape)

print("\nRépartition de la cible :")
print(
    labeled_transactions["is_fraud"]
    .value_counts()
    .sort_index()
)

print("\nPourcentage de fraude :")
print(
    labeled_transactions["is_fraud"]
    .value_counts(normalize=True)
    .mul(100)
    .round(4)
)

In [ ]:
print("\nContrôles :")

print(
    "Transactions labellisées :",
    len(labeled_transactions)
)

print(
    "IDs uniques :",
    labeled_transactions["id"].nunique()
)

print(
    "Dates invalides :",
    labeled_transactions["date"].isna().sum()
)

print(
    "Montants invalides :",
    labeled_transactions["amount_numeric"].isna().sum()
)

In [ ]:
# ============================================================
# 06 — DIAGNOSTIC GLOBAL DE LA TABLE LABELLISÉE
# ============================================================

print("=== DIMENSIONS ET IDENTIFIANTS ===")

print(
    "Transactions totales dans le dataset :",
    len(transactions)
)

print(
    "Transactions labellisées :",
    len(labeled_transactions)
)

print(
    "Transactions sans label :",
    len(transactions) - len(labeled_transactions)
)

print(
    "IDs de transactions uniques :",
    labeled_transactions["id"].nunique()
)


print("\n=== PÉRIODE COUVERTE ===")

print(
    "Date minimale :",
    labeled_transactions["date"].min()
)

print(
    "Date maximale :",
    labeled_transactions["date"].max()
)


print("\n=== VALEURS MANQUANTES ===")

missing_values = (
    labeled_transactions
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_values[missing_values > 0]
)


print("\n=== TYPES DE DONNÉES ===")

display(
    labeled_transactions.dtypes
)

In [ ]:
# ============================================================
# 07 — ANALYSE DE LA CIBLE ET DU DÉSÉQUILIBRE
# ============================================================

target_distribution = (
    labeled_transactions["is_fraud"]
    .value_counts()
    .sort_index()
)

target_percentage = (
    labeled_transactions["is_fraud"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(4)
)

print("=== RÉPARTITION DE LA CIBLE ===\n")

display(
    pd.DataFrame({
        "count": target_distribution,
        "percentage": target_percentage
    })
)

print("\n=== TAUX GLOBAL DE FRAUDE ===")

fraud_rate = (
    labeled_transactions["is_fraud"].mean() * 100
)

print(f"{fraud_rate:.4f} %")

print("\n=== RAPPORT NORMAL / FRAUDE ===")

normal_count = target_distribution.get(0, 0)
fraud_count = target_distribution.get(1, 0)

if fraud_count > 0:
    print(f"{normal_count / fraud_count:,.1f} transactions normales pour 1 fraude")

In [ ]:
# ============================================================
# 08 — ANALYSE TEMPORELLE DE BASE
# ============================================================

# Création des variables temporelles
labeled_transactions["year"] = (
    labeled_transactions["date"].dt.year
)

labeled_transactions["month"] = (
    labeled_transactions["date"].dt.month
)

labeled_transactions["transaction_hour"] = (
    labeled_transactions["date"].dt.hour
)

labeled_transactions["day_of_week"] = (
    labeled_transactions["date"].dt.dayofweek
)

labeled_transactions["is_weekend"] = (
    labeled_transactions["day_of_week"] >= 5
).astype("int8")

labeled_transactions["is_night"] = (
    (labeled_transactions["transaction_hour"] < 6)
    | (labeled_transactions["transaction_hour"] >= 22)
).astype("int8")


print("=== RÉPARTITION PAR ANNÉE ===\n")

print(
    labeled_transactions["year"]
    .value_counts()
    .sort_index()
)

print("\n=== RÉPARTITION PAR CANAL ===\n")

print(
    labeled_transactions["use_chip"]
    .value_counts(dropna=False)
)

print("\n=== RÉPARTITION PAR HEURE ===\n")

print(
    labeled_transactions["transaction_hour"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ------------------------------------------------------------
# Taux de fraude selon le canal
# ------------------------------------------------------------

fraud_by_channel = pd.crosstab(
    labeled_transactions["use_chip"],
    labeled_transactions["is_fraud"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

fraud_by_channel["fraud_rate"] = (
    fraud_by_channel[1]
    / fraud_by_channel[[0, 1]].sum(axis=1)
    * 100
)

print("\n=== FRAUDE PAR CANAL ===\n")
display(fraud_by_channel)


# ------------------------------------------------------------
# Taux de fraude selon le jour/week-end
# ------------------------------------------------------------

fraud_by_weekend = pd.crosstab(
    labeled_transactions["is_weekend"],
    labeled_transactions["is_fraud"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

fraud_by_weekend["fraud_rate"] = (
    fraud_by_weekend[1]
    / fraud_by_weekend[[0, 1]].sum(axis=1)
    * 100
)

print("\n=== FRAUDE WEEK-END / SEMAINE ===\n")
display(fraud_by_weekend)


# ------------------------------------------------------------
# Taux de fraude selon la nuit
# ------------------------------------------------------------

fraud_by_night = pd.crosstab(
    labeled_transactions["is_night"],
    labeled_transactions["is_fraud"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

fraud_by_night["fraud_rate"] = (
    fraud_by_night[1]
    / fraud_by_night[[0, 1]].sum(axis=1)
    * 100
)

print("\n=== FRAUDE JOUR / NUIT ===\n")
display(fraud_by_night)


# ------------------------------------------------------------
# Taux de fraude selon l'heure
# ------------------------------------------------------------

fraud_by_hour = pd.crosstab(
    labeled_transactions["transaction_hour"],
    labeled_transactions["is_fraud"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

fraud_by_hour["fraud_rate"] = (
    fraud_by_hour[1]
    / fraud_by_hour[[0, 1]].sum(axis=1)
    * 100
)

print("\n=== FRAUDE PAR HEURE ===\n")
display(fraud_by_hour)

In [ ]:
# ============================================================
# 09 — ANALYSE DES ERREURS
# ============================================================

error_types = {
    "bad_pin": "Bad PIN",
    "bad_cvv": "Bad CVV",
    "bad_card_number": "Bad Card Number",
    "bad_expiration": "Bad Expiration",
    "bad_zipcode": "Bad Zipcode",
    "insufficient_balance": "Insufficient Balance",
    "technical_glitch": "Technical Glitch"
}

# ------------------------------------------------------------
# Création des indicateurs binaires
# ------------------------------------------------------------

for feature_name, error_text in error_types.items():
    labeled_transactions[feature_name] = (
        labeled_transactions["errors"]
        .fillna("")
        .str.contains(
            error_text,
            regex=False
        )
        .astype("int8")
    )

# Nombre de types d'erreurs présents dans chaque transaction
error_columns = list(error_types.keys())

labeled_transactions["error_count"] = (
    labeled_transactions[error_columns]
    .sum(axis=1)
    .astype("int8")
)

# ------------------------------------------------------------
# Erreur présente ou non
# ------------------------------------------------------------

fraud_by_error = pd.crosstab(
    labeled_transactions["has_error"],
    labeled_transactions["is_fraud"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

fraud_by_error["fraud_rate"] = (
    fraud_by_error[1]
    / fraud_by_error[[0, 1]].sum(axis=1)
    * 100
)

print("=== FRAUDE SELON LA PRÉSENCE D'UNE ERREUR ===\n")
display(fraud_by_error)

# ------------------------------------------------------------
# Taux de fraude par type d'erreur
# ------------------------------------------------------------

error_fraud_rates = []

for feature_name in error_types:

    temp = pd.crosstab(
        labeled_transactions[feature_name],
        labeled_transactions["is_fraud"]
    ).reindex(
        columns=[0, 1],
        fill_value=0
    )

    if 1 in temp.index:
        total_with_error = temp.loc[1, [0, 1]].sum()

        fraud_rate = (
            temp.loc[1, 1] / total_with_error * 100
            if total_with_error > 0
            else np.nan
        )

        fraud_count = temp.loc[1, 1]
    else:
        fraud_rate = np.nan
        fraud_count = 0

    error_fraud_rates.append(
        [
            feature_name,
            fraud_rate,
            fraud_count
        ]
    )

error_fraud_rates_df = pd.DataFrame(
    error_fraud_rates,
    columns=[
        "error_type",
        "fraud_rate_when_present",
        "fraud_count"
    ]
).sort_values(
    "fraud_rate_when_present",
    ascending=False
)

print("\n=== TAUX DE FRAUDE PAR TYPE D'ERREUR ===\n")
display(error_fraud_rates_df)

# ------------------------------------------------------------
# Nombre d'erreurs par transaction
# ------------------------------------------------------------

print("\n=== NOMBRE DE TYPES D'ERREURS ===\n")
display(
    labeled_transactions["error_count"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# 10 — PRÉPARATION ET ANALYSE DES CARTES
# ============================================================

cards_eda = cards[
    [
        "id",
        "client_id",
        "card_brand",
        "card_type",
        "has_chip",
        "credit_limit",
        "card_on_dark_web"
    ]
].copy()

# Harmoniser le nom de la clé
cards_eda = cards_eda.rename(
    columns={"id": "card_id"}
)

# ------------------------------------------------------------
# Conversion de la limite de crédit
# ------------------------------------------------------------

cards_eda["credit_limit_numeric"] = (
    cards_eda["credit_limit"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# ------------------------------------------------------------
# Jointure transactions ↔ cartes
# ------------------------------------------------------------

transactions_with_cards = labeled_transactions.merge(
    cards_eda,
    on="card_id",
    how="left",
    validate="many_to_one"
)

print("=== CONTRÔLE DE LA JOINTURE ===")
print("Transactions labellisées :", len(labeled_transactions))
print("Après jointure            :", len(transactions_with_cards))
print(
    "IDs uniques               :",
    transactions_with_cards["id"].nunique()
)

print("\nValeurs manquantes après jointure :")
display(
    transactions_with_cards[
        [
            "card_brand",
            "card_type",
            "has_chip",
            "credit_limit_numeric",
            "card_on_dark_web"
        ]
    ]
    .isna()
    .sum()
)

In [ ]:
# ------------------------------------------------------------
# Fraude selon le type de carte
# ------------------------------------------------------------

fraud_by_card_type = pd.crosstab(
    transactions_with_cards["card_type"],
    transactions_with_cards["is_fraud"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

fraud_by_card_type["fraud_rate"] = (
    fraud_by_card_type[1]
    / fraud_by_card_type[[0, 1]].sum(axis=1)
    * 100
)

fraud_by_card_type = fraud_by_card_type.sort_values(
    "fraud_rate",
    ascending=False
)

print("\n=== FRAUDE PAR TYPE DE CARTE ===")
display(fraud_by_card_type)


# ------------------------------------------------------------
# Fraude selon la présence d'une puce
# ------------------------------------------------------------

fraud_by_has_chip = pd.crosstab(
    transactions_with_cards["has_chip"],
    transactions_with_cards["is_fraud"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

fraud_by_has_chip["fraud_rate"] = (
    fraud_by_has_chip[1]
    / fraud_by_has_chip[[0, 1]].sum(axis=1)
    * 100
)

print("\n=== FRAUDE SELON HAS_CHIP ===")
display(fraud_by_has_chip)


# ------------------------------------------------------------
# Fraude selon la présence sur le dark web
# ------------------------------------------------------------

fraud_by_dark_web = pd.crosstab(
    transactions_with_cards["card_on_dark_web"],
    transactions_with_cards["is_fraud"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

fraud_by_dark_web["fraud_rate"] = (
    fraud_by_dark_web[1]
    / fraud_by_dark_web[[0, 1]].sum(axis=1)
    * 100
)

print("\n=== FRAUDE SELON CARD_ON_DARK_WEB ===")
display(fraud_by_dark_web)

In [ ]:
# ------------------------------------------------------------
# Ratio montant / limite de crédit
# ------------------------------------------------------------

transactions_with_cards["amount_to_credit_limit"] = np.where(
    transactions_with_cards["credit_limit_numeric"] > 0,
    transactions_with_cards["amount_numeric"]
    / transactions_with_cards["credit_limit_numeric"],
    np.nan
)

# Supprimer d'éventuelles valeurs infinies
transactions_with_cards["amount_to_credit_limit"] = (
    transactions_with_cards["amount_to_credit_limit"]
    .replace([np.inf, -np.inf], np.nan)
)

print("\n=== MONTANT / LIMITE DE CRÉDIT ===")

display(
    transactions_with_cards.groupby("is_fraud")[
        "amount_to_credit_limit"
    ].describe()
)

In [ ]:
# ============================================================
# 11 — PRÉPARATION ET ANALYSE DU PROFIL CLIENT
# ============================================================

clients_eda = users[
    [
        "id",
        "current_age",
        "retirement_age",
        "birth_year",
        "birth_month",
        "gender",
        "latitude",
        "longitude",
        "per_capita_income",
        "yearly_income",
        "total_debt",
        "credit_score",
        "num_credit_cards"
    ]
].copy()

# Harmoniser la clé avec les transactions
clients_eda = clients_eda.rename(
    columns={"id": "client_id"}
)

# ------------------------------------------------------------
# Conversion des variables monétaires
# ------------------------------------------------------------

for col in [
    "per_capita_income",
    "yearly_income",
    "total_debt"
]:
    clients_eda[f"{col}_numeric"] = (
        clients_eda[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

# ------------------------------------------------------------
# Jointure transactions ↔ clients
# ------------------------------------------------------------

transactions_with_clients = labeled_transactions.merge(
    clients_eda[
        [
            "client_id",
            "current_age",
            "retirement_age",
            "birth_year",
            "birth_month",
            "gender",
            "latitude",
            "longitude",
            "per_capita_income_numeric",
            "yearly_income_numeric",
            "total_debt_numeric",
            "credit_score",
            "num_credit_cards"
        ]
    ],
    on="client_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# Contrôle de la jointure
# ------------------------------------------------------------

print("=== CONTRÔLE DE LA JOINTURE ===")

print(
    "Transactions labellisées :",
    len(labeled_transactions)
)

print(
    "Après jointure            :",
    len(transactions_with_clients)
)

print(
    "IDs uniques               :",
    transactions_with_clients["id"].nunique()
)

print("\nValeurs manquantes :")

display(
    transactions_with_clients[
        [
            "current_age",
            "yearly_income_numeric",
            "total_debt_numeric",
            "credit_score",
            "num_credit_cards"
        ]
    ]
    .isna()
    .sum()
)

In [ ]:
# ------------------------------------------------------------
# Ratios financiers
# ------------------------------------------------------------

transactions_with_clients["debt_to_income"] = np.where(
    transactions_with_clients["yearly_income_numeric"] > 0,
    transactions_with_clients["total_debt_numeric"]
    / transactions_with_clients["yearly_income_numeric"],
    np.nan
)

transactions_with_clients["amount_to_income"] = np.where(
    transactions_with_clients["yearly_income_numeric"] > 0,
    transactions_with_clients["amount_numeric"]
    / transactions_with_clients["yearly_income_numeric"],
    np.nan
)

# Éviter les valeurs infinies
for col in [
    "debt_to_income",
    "amount_to_income"
]:
    transactions_with_clients[col] = (
        transactions_with_clients[col]
        .replace([np.inf, -np.inf], np.nan)
    )

# ------------------------------------------------------------
# Statistiques du profil client
# ------------------------------------------------------------

client_features = [
    "current_age",
    "yearly_income_numeric",
    "total_debt_numeric",
    "credit_score",
    "num_credit_cards"
]

print("\n=== STATISTIQUES DU PROFIL CLIENT ===")
display(
    transactions_with_clients[
        client_features
    ].describe().T
)

print("\n=== MOYENNES : FRAUDE VS NON FRAUDE ===")
display(
    transactions_with_clients
    .groupby("is_fraud")[client_features]
    .mean()
    .T
)

print("\n=== DETTE / REVENU ===")
display(
    transactions_with_clients
    .groupby("is_fraud")["debt_to_income"]
    .describe()
)

print("\n=== MONTANT / REVENU ===")
display(
    transactions_with_clients
    .groupby("is_fraud")["amount_to_income"]
    .describe()
)

In [ ]:
# ------------------------------------------------------------
# Fraude selon la tranche de score de crédit
# ------------------------------------------------------------

transactions_with_clients["credit_score_band"] = pd.cut(
    transactions_with_clients["credit_score"],
    bins=[0, 580, 670, 740, 800, 850],
    labels=[
        "Très faible",
        "Faible",
        "Bon",
        "Très bon",
        "Excellent"
    ],
    include_lowest=True
)

fraud_by_credit_score = pd.crosstab(
    transactions_with_clients["credit_score_band"],
    transactions_with_clients["is_fraud"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

fraud_by_credit_score["fraud_rate"] = (
    fraud_by_credit_score[1]
    / fraud_by_credit_score[[0, 1]].sum(axis=1)
    * 100
)

print("\n=== FRAUDE PAR SCORE DE CRÉDIT ===")
display(fraud_by_credit_score)


# ------------------------------------------------------------
# Fraude selon l'âge
# ------------------------------------------------------------

transactions_with_clients["age_band"] = pd.cut(
    transactions_with_clients["current_age"],
    bins=[0, 25, 35, 50, 65, 100],
    labels=[
        "<=25",
        "26-35",
        "36-50",
        "51-65",
        ">65"
    ],
    include_lowest=True
)

fraud_by_age = pd.crosstab(
    transactions_with_clients["age_band"],
    transactions_with_clients["is_fraud"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

fraud_by_age["fraud_rate"] = (
    fraud_by_age[1]
    / fraud_by_age[[0, 1]].sum(axis=1)
    * 100
)

print("\n=== FRAUDE PAR TRANCHE D'ÂGE ===")
display(fraud_by_age)

In [ ]:
# ============================================================
# 12 — BASE HISTORIQUE COMPLÈTE POUR LES FEATURES
# ============================================================

# Toutes les transactions, y compris celles qui n'ont pas
# de label de fraude.
transaction_history = transactions_eda.copy()

# ------------------------------------------------------------
# Ajouter les informations client
# ------------------------------------------------------------

transaction_history = transaction_history.merge(
    clients_eda[
        [
            "client_id",
            "current_age",
            "yearly_income_numeric",
            "total_debt_numeric",
            "credit_score",
            "num_credit_cards"
        ]
    ],
    on="client_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# Ajouter les informations carte
# ------------------------------------------------------------

transaction_history = transaction_history.merge(
    cards_eda[
        [
            "card_id",
            "card_brand",
            "card_type",
            "has_chip",
            "credit_limit_numeric",
            "card_on_dark_web"
        ]
    ],
    on="card_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# Garantir un ordre temporel stable
# ------------------------------------------------------------

transaction_history = (
    transaction_history
    .sort_values(["client_id", "date", "id"])
    .reset_index(drop=True)
)

print("=== BASE HISTORIQUE ===")
print("Dimensions :", transaction_history.shape)

print("\nTransactions uniques :",
      transaction_history["id"].nunique())

print("\nClients uniques :",
      transaction_history["client_id"].nunique())

print("\nCartes uniques :",
      transaction_history["card_id"].nunique())

print("\nDates invalides :",
      transaction_history["date"].isna().sum())

In [ ]:
# ============================================================
# 13 — FEATURES COMPORTEMENTALES HISTORIQUES
# ============================================================

# ------------------------------------------------------------
# Historique de la transaction précédente du client
# ------------------------------------------------------------

transaction_history["previous_amount"] = (
    transaction_history
    .groupby("client_id")["amount_numeric"]
    .shift(1)
)

transaction_history["previous_transaction_time"] = (
    transaction_history
    .groupby("client_id")["date"]
    .shift(1)
)

# Temps écoulé depuis la transaction précédente (en heures)
transaction_history["time_since_previous_transaction"] = (
    transaction_history["date"]
    - transaction_history["previous_transaction_time"]
).dt.total_seconds() / 3600

# ------------------------------------------------------------
# Écart avec la transaction précédente
# ------------------------------------------------------------

transaction_history["amount_difference"] = (
    transaction_history["amount_numeric"]
    - transaction_history["previous_amount"]
).abs()

previous_amount_abs = (
    transaction_history["previous_amount"].abs()
)

transaction_history["amount_change_ratio"] = np.where(
    previous_amount_abs > 0,
    transaction_history["amount_difference"]
    / previous_amount_abs,
    np.nan
)

# ------------------------------------------------------------
# Montant moyen historique du client
# IMPORTANT :
# shift(1) exclut la transaction actuelle
# ------------------------------------------------------------

transaction_history["client_previous_avg_amount"] = (
    transaction_history
    .groupby("client_id")["amount_numeric"]
    .transform(
        lambda s: s.shift(1).expanding().mean()
    )
)

# ------------------------------------------------------------
# Écart du montant actuel par rapport à l'habitude du client
# ------------------------------------------------------------

transaction_history["amount_deviation_from_client"] = np.where(
    transaction_history["client_previous_avg_amount"].notna()
    & (transaction_history["client_previous_avg_amount"] != 0),

    (
        transaction_history["amount_numeric"]
        - transaction_history["client_previous_avg_amount"]
    ).abs()
    / transaction_history["client_previous_avg_amount"].abs(),

    np.nan
)

print("Features comportementales de base créées.")

print("\nNouvelles colonnes :")
print([
    "previous_amount",
    "previous_transaction_time",
    "time_since_previous_transaction",
    "amount_difference",
    "amount_change_ratio",
    "client_previous_avg_amount",
    "amount_deviation_from_client"
])

In [ ]:
print("\n=== EXEMPLE ===")

display(
    transaction_history[
        [
            "client_id",
            "date",
            "amount_numeric",
            "previous_amount",
            "previous_transaction_time",
            "time_since_previous_transaction",
            "client_previous_avg_amount",
            "amount_deviation_from_client"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# 14 — FEATURES DE VÉLOCITÉ DU CLIENT
# ============================================================

# Sécurité : tri temporel par client
transaction_history = (
    transaction_history
    .sort_values(["client_id", "date", "id"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Nombre de transactions du client dans les 24h précédentes
# ------------------------------------------------------------

transaction_history["client_transaction_count_24h"] = (
    transaction_history
    .groupby("client_id", group_keys=False)
    .apply(
        lambda g: (
            g.set_index("date")["id"]
            .rolling("24h", closed="left")
            .count()
        )
    )
    .reset_index(level=0, drop=True)
    .sort_index()
)

# ------------------------------------------------------------
# Nombre de transactions du client dans l'heure précédente
# ------------------------------------------------------------

transaction_history["client_transaction_count_1h"] = (
    transaction_history
    .groupby("client_id", group_keys=False)
    .apply(
        lambda g: (
            g.set_index("date")["id"]
            .rolling("1h", closed="left")
            .count()
        )
    )
    .reset_index(level=0, drop=True)
    .sort_index()
)

print("Features de vélocité créées.")

In [ ]:
print("\n=== VÉRIFICATION ===")

display(
    transaction_history[
        [
            "client_id",
            "date",
            "client_transaction_count_1h",
            "client_transaction_count_24h"
        ]
    ].head(20)
)

In [ ]:
print("\nValeurs manquantes :")

display(
    transaction_history[
        [
            "client_transaction_count_1h",
            "client_transaction_count_24h"
        ]
    ].isna().sum()
)

In [ ]:
# ============================================================
# 15 — MONTANT CUMULÉ DU CLIENT SUR 24H
# ============================================================

transaction_history["client_amount_24h"] = (
    transaction_history
    .groupby("client_id", group_keys=False)
    .apply(
        lambda g: (
            g.set_index("date")["amount_numeric"]
            .rolling(
                "24h",
                closed="left"
            )
            .sum()
        )
    )
    .reset_index(level=0, drop=True)
    .sort_index()
)

print("Feature client_amount_24h créée.")

display(
    transaction_history[
        [
            "client_id",
            "date",
            "amount_numeric",
            "client_amount_24h"
        ]
    ].head(20)
)

In [ ]:
print("\nValeurs manquantes :")

print(
    transaction_history["client_amount_24h"]
    .isna()
    .sum()
)

In [ ]:
transaction_history["amount_abs"] = (
    transaction_history["amount_numeric"].abs()
)

transaction_history["client_absolute_amount_24h"] = (
    transaction_history
    .groupby("client_id", group_keys=False)
    .apply(
        lambda g: (
            g.set_index("date")["amount_abs"]
            .rolling(
                "24h",
                closed="left"
            )
            .sum()
        )
    )
    .reset_index(level=0, drop=True)
    .sort_index()
)

print(
    "\nFeatures créées :",
    [
        "client_amount_24h",
        "client_absolute_amount_24h"
    ]
)

In [ ]:
# ============================================================
# 16 — FEATURES COMPORTEMENTALES DE LA CARTE
# ============================================================

# Sécurité : ordre temporel par carte
transaction_history = (
    transaction_history
    .sort_values(["card_id", "date", "id"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Nombre de transactions de la carte dans les 24h précédentes
# ------------------------------------------------------------

transaction_history["card_transaction_count_24h"] = (
    transaction_history
    .groupby("card_id", group_keys=False)
    .apply(
        lambda g: (
            g.set_index("date")["id"]
            .rolling(
                "24h",
                closed="left"
            )
            .count()
        )
    )
    .reset_index(level=0, drop=True)
    .sort_index()
)

# ------------------------------------------------------------
# Montant cumulé de la carte dans les 24h précédentes
# ------------------------------------------------------------

transaction_history["card_amount_24h"] = (
    transaction_history
    .groupby("card_id", group_keys=False)
    .apply(
        lambda g: (
            g.set_index("date")["amount_numeric"]
            .rolling(
                "24h",
                closed="left"
            )
            .sum()
        )
    )
    .reset_index(level=0, drop=True)
    .sort_index()
)

print("Features comportementales de la carte créées.")

display(
    transaction_history[
        [
            "card_id",
            "date",
            "amount_numeric",
            "card_transaction_count_24h",
            "card_amount_24h"
        ]
    ].head(20)
)

In [ ]:
print("\n=== VALEURS MANQUANTES ===")

display(
    transaction_history[
        [
            "card_transaction_count_24h",
            "card_amount_24h"
        ]
    ].isna().sum()
)

In [ ]:
# ============================================================
# 17 — FEATURES COMPORTEMENTALES DU MARCHAND
# ============================================================

# ------------------------------------------------------------
# Préparer les données nécessaires
# ------------------------------------------------------------

merchant_history = transaction_history[
    [
        "id",
        "merchant_id",
        "date",
        "amount_numeric"
    ]
].copy()

merchant_history = merchant_history.dropna(
    subset=["merchant_id", "date"]
)

merchant_history = merchant_history.sort_values(
    ["merchant_id", "date", "id"]
)

# ------------------------------------------------------------
# Fonctions de calcul
# ------------------------------------------------------------

def merchant_rolling_features(group):
    indexed = group.set_index("date")

    result = pd.DataFrame(index=group.index)

    result["merchant_transaction_count_24h"] = (
        indexed["id"]
        .rolling("24h", closed="left")
        .count()
        .to_numpy()
    )

    result["merchant_amount_24h"] = (
        indexed["amount_numeric"]
        .rolling("24h", closed="left")
        .sum()
        .to_numpy()
    )

    return result

# ------------------------------------------------------------
# Calcul
# ------------------------------------------------------------

merchant_features = (
    merchant_history
    .groupby("merchant_id", group_keys=False)
    .apply(merchant_rolling_features)
)

merchant_features = merchant_features.reset_index(
    level=0,
    drop=True
)

# Récupérer l'identifiant de transaction
merchant_features["id"] = merchant_history.loc[
    merchant_features.index,
    "id"
].values

# ------------------------------------------------------------
# Ajouter les features à la table historique
# ------------------------------------------------------------

transaction_history = transaction_history.merge(
    merchant_features[
        [
            "id",
            "merchant_transaction_count_24h",
            "merchant_amount_24h"
        ]
    ],
    on="id",
    how="left",
    validate="one_to_one"
)

print("Features comportementales du marchand créées.")

display(
    transaction_history[
        [
            "merchant_id",
            "date",
            "amount_numeric",
            "merchant_transaction_count_24h",
            "merchant_amount_24h"
        ]
    ].head(20)
)

In [ ]:
print("\n=== VALEURS MANQUANTES ===")

display(
    transaction_history[
        [
            "merchant_transaction_count_24h",
            "merchant_amount_24h"
        ]
    ].isna().sum()
)

In [ ]:
# ============================================================
# 18 — BASE HISTORIQUE LABELLISÉE POUR L'EDA / ML
# ============================================================

model_data = transaction_history.merge(
    labels_eda,
    left_on="id",
    right_index=True,
    how="inner",
    validate="one_to_one"
)

print("=== CONTRÔLE ===")

print(
    "Transactions historiques :",
    len(transaction_history)
)

print(
    "Transactions avec label   :",
    len(model_data)
)

print(
    "IDs uniques               :",
    model_data["id"].nunique()
)

print("\nRépartition de la cible :")

display(
    model_data["is_fraud"]
    .value_counts()
    .sort_index()
)

In [ ]:
print("\n=== VÉRIFICATION DES DOUBLONS ===")

duplicate_ids = (
    model_data["id"]
    .duplicated()
    .sum()
)

print(
    "Transactions dupliquées :",
    duplicate_ids
)

In [ ]:
# ============================================================
# 19 — ANALYSE DES FEATURES COMPORTEMENTALES
# ============================================================

behavior_features = [
    "amount_difference",
    "amount_change_ratio",
    "client_previous_avg_amount",
    "amount_deviation_from_client",
    "time_since_previous_transaction",
    "client_transaction_count_1h",
    "client_transaction_count_24h",
    "client_amount_24h",
    "client_absolute_amount_24h",
    "card_transaction_count_24h",
    "card_amount_24h",
    "merchant_transaction_count_24h",
    "merchant_amount_24h"
]

print("=== STATISTIQUES DES FEATURES COMPORTEMENTALES ===\n")

display(
    model_data
    .groupby("is_fraud")[behavior_features]
    .mean()
    .T
)

In [ ]:
print("\n=== STATISTIQUES DÉTAILLÉES ===")

for feature in behavior_features:
    print(f"\n--- {feature} ---")

    display(
        model_data
        .groupby("is_fraud")[feature]
        .describe()
    )

In [ ]:
# ============================================================
# 20 — CONTRÔLE FINAL DES FEATURES
# ============================================================

# Features numériques principales
numeric_features = [
    "amount_numeric",
    "current_age",
    "yearly_income_numeric",
    "total_debt_numeric",
    "credit_score",
    "num_credit_cards",
    "debt_to_income",
    "amount_to_income",
    "amount_to_credit_limit",
    "time_since_previous_transaction",
    "amount_difference",
    "amount_change_ratio",
    "client_previous_avg_amount",
    "amount_deviation_from_client",
    "client_transaction_count_1h",
    "client_transaction_count_24h",
    "client_amount_24h",
    "client_absolute_amount_24h",
    "card_transaction_count_24h",
    "card_amount_24h",
    "merchant_transaction_count_24h",
    "merchant_amount_24h"
]

# Garder uniquement les colonnes réellement présentes
numeric_features = [
    col
    for col in numeric_features
    if col in model_data.columns
]

# ------------------------------------------------------------
# Valeurs manquantes
# ------------------------------------------------------------

missing_summary = (
    model_data[numeric_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = (
    missing_summary[missing_summary > 0]
    .to_frame("missing_count")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"]
    / len(model_data)
    * 100
)

print("=== VALEURS MANQUANTES ===")

display(missing_summary)


# ------------------------------------------------------------
# Valeurs infinies
# ------------------------------------------------------------

infinite_summary = pd.Series(
    {
        col: np.isinf(
            model_data[col].dropna()
        ).sum()
        for col in numeric_features
    },
    name="infinite_count"
)

print("\n=== VALEURS INFINIES ===")

display(
    infinite_summary[
        infinite_summary > 0
    ]
)


# ------------------------------------------------------------
# Statistiques générales
# ------------------------------------------------------------

print("\n=== STATISTIQUES DES VARIABLES NUMÉRIQUES ===")

display(
    model_data[numeric_features]
    .describe()
    .T
)

In [ ]:
# ============================================================
# 21 — SYNTHÈSE FINALE DE L'EDA
# ============================================================

# ------------------------------------------------------------
# Variables transactionnelles
# ------------------------------------------------------------

transaction_features = [
    "amount_numeric",
    "use_chip",
    "merchant_id",
    "merchant_city",
    "merchant_state",
    "mcc",
    "has_error",
    "error_count",
    "bad_pin",
    "bad_cvv",
    "bad_card_number",
    "bad_expiration",
    "bad_zipcode",
    "insufficient_balance",
    "technical_glitch"
]

# ------------------------------------------------------------
# Variables client
# ------------------------------------------------------------

client_features = [
    "current_age",
    "yearly_income_numeric",
    "total_debt_numeric",
    "credit_score",
    "num_credit_cards",
    "debt_to_income",
    "amount_to_income"
]

# ------------------------------------------------------------
# Variables carte
# ------------------------------------------------------------

card_features = [
    "card_brand",
    "card_type",
    "has_chip",
    "credit_limit_numeric",
    "card_on_dark_web",
    "amount_to_credit_limit"
]

# ------------------------------------------------------------
# Variables temporelles
# ------------------------------------------------------------

temporal_features = [
    "year",
    "month",
    "transaction_hour",
    "day_of_week",
    "is_weekend",
    "is_night"
]

# ------------------------------------------------------------
# Variables comportementales
# ------------------------------------------------------------

behavior_features = [
    "previous_amount",
    "time_since_previous_transaction",
    "amount_difference",
    "amount_change_ratio",
    "client_previous_avg_amount",
    "amount_deviation_from_client",
    "client_transaction_count_1h",
    "client_transaction_count_24h",
    "client_amount_24h",
    "client_absolute_amount_24h",
    "card_transaction_count_24h",
    "card_amount_24h",
    "merchant_transaction_count_24h",
    "merchant_amount_24h"
]

# ------------------------------------------------------------
# Vérifier la présence des variables
# ------------------------------------------------------------

feature_groups = {
    "Transactionnelles": transaction_features,
    "Client": client_features,
    "Carte": card_features,
    "Temporelles": temporal_features,
    "Comportementales": behavior_features
}

summary_rows = []

for group_name, features in feature_groups.items():

    for feature in features:

        summary_rows.append({
            "group": group_name,
            "feature": feature,
            "present": feature in model_data.columns
        })

feature_summary = pd.DataFrame(summary_rows)

print("=== SYNTHÈSE DES FEATURES ===")

display(feature_summary)

In [ ]:
# ============================================================
# 22 — CONSTRUCTION FINALE DE MODEL_DATA
# ============================================================

# ------------------------------------------------------------
# Variables temporelles
# ------------------------------------------------------------

transaction_history["year"] = (
    transaction_history["date"].dt.year
)

transaction_history["month"] = (
    transaction_history["date"].dt.month
)

transaction_history["transaction_hour"] = (
    transaction_history["date"].dt.hour
)

transaction_history["day_of_week"] = (
    transaction_history["date"].dt.dayofweek
)

transaction_history["is_weekend"] = (
    transaction_history["day_of_week"] >= 5
).astype("int8")

transaction_history["is_night"] = (
    (transaction_history["transaction_hour"] < 6)
    | (transaction_history["transaction_hour"] >= 22)
).astype("int8")


# ------------------------------------------------------------
# Ratios financiers
# ------------------------------------------------------------

transaction_history["debt_to_income"] = np.where(
    transaction_history["yearly_income_numeric"] > 0,
    transaction_history["total_debt_numeric"]
    / transaction_history["yearly_income_numeric"],
    np.nan
)

transaction_history["amount_to_income"] = np.where(
    transaction_history["yearly_income_numeric"] > 0,
    transaction_history["amount_numeric"]
    / transaction_history["yearly_income_numeric"],
    np.nan
)

transaction_history["amount_to_credit_limit"] = np.where(
    transaction_history["credit_limit_numeric"] > 0,
    transaction_history["amount_numeric"]
    / transaction_history["credit_limit_numeric"],
    np.nan
)

# Éliminer les valeurs infinies
ratio_columns = [
    "debt_to_income",
    "amount_to_income",
    "amount_to_credit_limit"
]

for col in ratio_columns:
    transaction_history[col] = (
        transaction_history[col]
        .replace([np.inf, -np.inf], np.nan)
    )


# ------------------------------------------------------------
# Variables d'erreur
# ------------------------------------------------------------

error_types = {
    "bad_pin": "Bad PIN",
    "bad_cvv": "Bad CVV",
    "bad_card_number": "Bad Card Number",
    "bad_expiration": "Bad Expiration",
    "bad_zipcode": "Bad Zipcode",
    "insufficient_balance": "Insufficient Balance",
    "technical_glitch": "Technical Glitch"
}

for feature_name, error_text in error_types.items():
    transaction_history[feature_name] = (
        transaction_history["errors"]
        .fillna("")
        .str.contains(
            error_text,
            regex=False
        )
        .astype("int8")
    )

error_columns = list(error_types.keys())

transaction_history["error_count"] = (
    transaction_history[error_columns]
    .sum(axis=1)
    .astype("int8")
)


# ------------------------------------------------------------
# Nettoyer les anciennes colonnes de label éventuelles
# ------------------------------------------------------------

columns_to_drop = [
    col
    for col in ["is_fraud"]
    if col in transaction_history.columns
]

if columns_to_drop:
    transaction_history = transaction_history.drop(
        columns=columns_to_drop
    )


# ------------------------------------------------------------
# Ajouter la cible une seule fois
# ------------------------------------------------------------

model_data = transaction_history.merge(
    labels_eda,
    left_on="id",
    right_index=True,
    how="inner",
    validate="one_to_one"
)


# ------------------------------------------------------------
# Contrôle final
# ------------------------------------------------------------

print("=== MODEL_DATA FINAL ===")
print("Dimensions :", model_data.shape)

print("\nTransactions uniques :",
      model_data["id"].nunique())

print("\nRépartition cible :")
display(
    model_data["is_fraud"]
    .value_counts()
    .sort_index()
)

print("\nNombre de colonnes :", len(model_data.columns))

In [ ]:
# ============================================================
# 23 — CONTRÔLE FINAL DES VARIABLES
# ============================================================

feature_groups = {
    "Transactionnelles": [
        "amount_numeric",
        "use_chip",
        "merchant_id",
        "merchant_city",
        "merchant_state",
        "zip",
        "mcc",
        "has_error",
        "error_count",
        "bad_pin",
        "bad_cvv",
        "bad_card_number",
        "bad_expiration",
        "bad_zipcode",
        "insufficient_balance",
        "technical_glitch"
    ],

    "Client": [
        "client_id",
        "current_age",
        "yearly_income_numeric",
        "total_debt_numeric",
        "credit_score",
        "num_credit_cards",
        "debt_to_income",
        "amount_to_income"
    ],

    "Carte": [
        "card_id",
        "card_brand",
        "card_type",
        "has_chip",
        "credit_limit_numeric",
        "card_on_dark_web",
        "amount_to_credit_limit"
    ],

    "Temporelles": [
        "year",
        "month",
        "transaction_hour",
        "day_of_week",
        "is_weekend",
        "is_night"
    ],

    "Comportementales": [
        "previous_amount",
        "previous_transaction_time",
        "time_since_previous_transaction",
        "amount_difference",
        "amount_change_ratio",
        "client_previous_avg_amount",
        "amount_deviation_from_client",
        "client_transaction_count_1h",
        "client_transaction_count_24h",
        "client_amount_24h",
        "client_absolute_amount_24h",
        "card_transaction_count_24h",
        "card_amount_24h",
        "merchant_transaction_count_24h",
        "merchant_amount_24h"
    ]
}

# ------------------------------------------------------------
# Vérification de présence
# ------------------------------------------------------------

rows = []

for group, features in feature_groups.items():
    for feature in features:
        rows.append({
            "group": group,
            "feature": feature,
            "present": feature in model_data.columns
        })

final_feature_check = pd.DataFrame(rows)

print("=== VARIABLES ABSENTES ===")

display(
    final_feature_check[
        ~final_feature_check["present"]
    ]
)

print("\n=== NOMBRE DE VARIABLES PAR GROUPE ===")

display(
    final_feature_check
    .groupby("group")["feature"]
    .count()
)

print("\n=== CONTRÔLE FINAL ===")

print("Nombre de lignes :", len(model_data))
print("Nombre de colonnes :", len(model_data.columns))

print(
    "Nombre de variables candidates :",
    len(final_feature_check)
)

print(
    "Variables candidates présentes :",
    final_feature_check["present"].sum()
)

In [ ]:
from pathlib import Path

# ============================================================
# EXPORT DE MODEL_DATA POUR LE NOTEBOOK DE PRÉPARATION
# ============================================================

# Définir le dossier de sortie
OUTPUT_DIR = Path("ml/data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Définir le chemin du fichier
MODEL_DATA_FILE = OUTPUT_DIR / "model_data.parquet"

# Exporter le DataFrame
MODEL_DATA.to_parquet(MODEL_DATA_FILE, index=False)

print(f"Fichier exporté avec succès : {MODEL_DATA_FILE}")
print(f"Dimensions : {MODEL_DATA.shape}")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent.parent

print("Dossier du projet :", PROJECT_DIR)

for dossier in [
    PROJECT_DIR / "ml" / "models",
    PROJECT_DIR / "ml" / "data" / "processed",
    PROJECT_DIR / "data" / "processed",
]:
    if dossier.exists():
        print(f"\nFichiers dans {dossier} :")
        for fichier in dossier.rglob("*"):
            if fichier.is_file() and any(
                mot in fichier.name.lower()
                for mot in ["xgb", "xgboost", "model"]
            ):
                print(fichier)

#  24  Galerie de visualisations exploratoires (EDA)

Ces graphiques complètent les analyses précédentes. Les taux de fraude sont calculés sur les transactions labellisées. Pour les histogrammes et les corrélations, un échantillon est utilisé afin de limiter la consommation mémoire. Les graphiques décrivent des associations, pas des relations causales.


In [ ]:
# 1. Déséquilibre de la cible
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

target_counts = model_data["is_fraud"].value_counts().reindex([0, 1], fill_value=0)
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(["Transactions légitimes", "Fraudes"], target_counts.values)
ax.set_title("Répartition des classes dans les données labellisées")
ax.set_ylabel("Nombre de transactions")
ax.ticklabel_format(axis="y", style="plain")
for bar, value in zip(bars, target_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{value:,.0f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()


In [ ]:
# 2. Évolution annuelle du volume et du taux de fraude
yearly = model_data.groupby("year", observed=True)["is_fraud"].agg(transactions="size", fraud_rate="mean").reset_index()
fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax1.bar(yearly["year"].astype(str), yearly["transactions"], alpha=0.75, label="Transactions")
ax1.set_ylabel("Nombre de transactions")
ax1.set_title("Volume des transactions et taux de fraude par année")
ax2 = ax1.twinx()
ax2.plot(yearly["year"].astype(str), yearly["fraud_rate"]*100, marker="o", linewidth=2, label="Taux de fraude (%)")
ax2.set_ylabel("Taux de fraude (%)")
ax1.set_xlabel("Année")
plt.tight_layout()
plt.show()


In [ ]:
# 3. Taux de fraude selon l'heure de la transaction
hourly = model_data.groupby("transaction_hour", observed=True)["is_fraud"].agg(transactions="size", fraud_rate="mean").reset_index()
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(hourly["transaction_hour"], hourly["fraud_rate"]*100, marker="o")
ax.set_title("Taux de fraude selon l'heure")
ax.set_xlabel("Heure de la journée")
ax.set_ylabel("Taux de fraude (%)")
ax.set_xticks(range(0, 24, 2))
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# 4. Taux de fraude par canal de paiement
if "use_chip" in model_data.columns:
    channel = model_data.groupby("use_chip", dropna=False)["is_fraud"].agg(transactions="size", fraud_rate="mean").sort_values("fraud_rate")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.barh(channel.index.astype(str), channel["fraud_rate"]*100)
    ax.set_title("Taux de fraude par canal de paiement")
    ax.set_xlabel("Taux de fraude (%)")
    for y, v in enumerate(channel["fraud_rate"]*100):
        ax.text(v, y, f" {v:.3f}%", va="center")
    plt.tight_layout()
    plt.show()


In [ ]:
# 5. Taux de fraude par type de carte
if "card_type" in model_data.columns:
    card_rates = model_data.groupby("card_type", dropna=False)["is_fraud"].agg(transactions="size", fraud_rate="mean")
    card_rates = card_rates.sort_values("fraud_rate").tail(15)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.barh(card_rates.index.astype(str), card_rates["fraud_rate"]*100)
    ax.set_title("Taux de fraude par type de carte (catégories les plus élevées)")
    ax.set_xlabel("Taux de fraude (%)")
    plt.tight_layout()
    plt.show()


In [ ]:
# 6. Distribution des montants par classe (échantillon pour la mémoire)
if "amount_numeric" in model_data.columns:
    sample_n = min(250_000, len(model_data))
    sample = model_data.sample(n=sample_n, random_state=42)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for label, name in [(0, "Légitime"), (1, "Fraude")]:
        values = sample.loc[sample["is_fraud"] == label, "amount_numeric"].dropna()
        values = values[values > 0]
        if len(values):
            ax.hist(values, bins=70, alpha=0.55, density=True, label=name)
    ax.set_xscale("log")
    ax.set_title("Distribution des montants positifs par classe (échantillon, axe X logarithmique)")
    ax.set_xlabel("Montant (échelle logarithmique)")
    ax.set_ylabel("Densité")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# 7. Taux de fraude selon le nombre d'erreurs
if "error_count" in model_data.columns:
    error_rates = model_data.groupby("error_count", dropna=False)["is_fraud"].agg(transactions="size", fraud_rate="mean").reset_index()
    error_rates = error_rates[error_rates["transactions"] >= 100].sort_values("error_count").head(20)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(error_rates["error_count"].astype(str), error_rates["fraud_rate"]*100)
    ax.set_title("Taux de fraude selon le nombre d'erreurs (catégories avec au moins 100 transactions)")
    ax.set_xlabel("Nombre d'erreurs")
    ax.set_ylabel("Taux de fraude (%)")
    plt.tight_layout()
    plt.show()


In [ ]:
# 8. Comparaison de variables comportementales : médianes
behavior_cols = [c for c in [
    "amount_difference", "client_transaction_count_1h",
    "client_transaction_count_24h", "card_transaction_count_24h",
    "merchant_transaction_count_24h", "time_since_previous_transaction"
] if c in model_data.columns]
if behavior_cols:
    medians = model_data.groupby("is_fraud")[behavior_cols].median().T
    medians.columns = ["Légitime", "Fraude"]
    ax = medians.plot(kind="bar", figsize=(11, 5))
    ax.set_title("Médianes des variables comportementales selon la classe")
    ax.set_ylabel("Médiane (échelle propre à chaque variable)")
    ax.set_xlabel("Variable")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# 9. Corrélations entre variables numériques principales (échantillon)
corr_cols = [c for c in [
    "amount_numeric", "credit_score", "current_age", "error_count",
    "client_transaction_count_1h", "client_transaction_count_24h",
    "card_transaction_count_24h", "merchant_transaction_count_24h",
    "amount_to_credit_limit", "is_fraud"
] if c in model_data.columns and pd.api.types.is_numeric_dtype(model_data[c])]
corr_sample = model_data[corr_cols].sample(n=min(100_000, len(model_data)), random_state=42)
corr = corr_sample.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=70, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
ax.set_title("Corrélations des variables numériques — échantillon")
fig.colorbar(im, ax=ax, label="Corrélation")
plt.tight_layout()
plt.show()
